<a href="https://colab.research.google.com/github/keilerdnietoc-rgb/colab/blob/main/ea1_ingesti_n_de_datos_desde_un_api__2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**<span style="color:#809bd8">Curso:</span>** **Introducción a Ingeniería de Sistemas**

**<span style="color:#809bd8">Tema:</span>** **Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Estudiante:</span>** **KEILER NIETO CARRILLO**

**<span style="color:#809bd8">Profesor:</span>** **Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>** **UNAC**

**<span style="color:#809bd8">Fecha:</span>** **11/03/2026**

In [ ]:
# pip install requests pandas openpyxl

In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
#una api es una  conjunto de peticiones y definiciones que permite a dos aplicaciones de software comunicarse e intercambiar datos entre sí de manera estandarizada
# desde la API https://jsonplaceholder.typicode.com/
API_URL = "https://jsonplaceholder.typicode.com/users"


# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
def extraer_datos(url):#realiza una peticion GET a la api y valida el codigo de estado.
    response = requests.get(url)#validando codigo 200 (exito) o lanzado excepcion en caso de error (ej. 404)
    if response.status_code == 200:#si es exitoso, convierte la respuesta JSON en una lista/diccionario de python
        return response.json()
    else: #si falla, lanza un aexepcion con el codigo de error recibido
        raise Exception(f"Error al conectar con la API: {response.status_code}")



# Creacion de la base de datos en SQLite "datos_api.db"
#una base de datos recopilación organizada de información o datos estructurados, almacenados electrónicamente en un sistema informático, que permite acceder, gestionar, modificar y actualizar la información fácilmente
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
def guardar_en_db(datos):#establece conexion con el archivo de base de datos(se crea si no existe)
    conn = sqlite3.connect('datos_api.db')#crea o abre una base de datos llamada (datos_api.db)
    cursor = conn.cursor()#el cursor permite ejecutar sentencias SQL

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')

    # Inserción de datos en la tabla "usuarios"
    for item in datos:#intercepta o reemplaza los datos en la tabla segun el ID
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    conn.commit()#aceptacion mediante "commit" para asegurar la persistencias de los datos.

    # Cerrado de la conexion a la base de datos "datos_api.db"
    conn.close()#cierra la conexion para liberar recursos conn.close

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
def generar_evidencias(datos_api):#abre conexion para leer los datos recien guardados
    # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')#usa pandas para leer la tabla SQL y convertirla en un dataframe
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)#ejecuta una consulta SQL y carga el resultado directamente en un dataframe de pandas
    df_db.to_csv('muestra_usuarios.csv', index=False)#exporta el contenido del dataframe a un archivo CSV
    conn.close()#cierra la conexion tras  la lectura

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    #obtiene la cantidad de registros desde la fuente (API) y el destino(Db)
    registros_api = len(datos_api)# cuenta cuantos elementos se recibieron originalmente de la api
    registros_db = len(df_db)#cuenta cuantas filas se leyeron desde la tabla de la base de datos

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    with open('auditoria.txt', 'w', encoding='utf-8') as f:#abre un archivo de texto en modo ecritura("w") con codificacion utf-8 para el reporte
        f.write("RESUMEN DE AUDITORÍA\n")#escribe encabezados y los resultados del conteo en el archivo de texto.
        f.write("====================\n")# escribe una linea divisora estetica en el archivo de un texto para separar el titulo del contenido
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")

        if registros_api == registros_db:#valida  la integridad de los datos comparando las fuentes
            f.write("\nESTADO: Operacion Exitosa.\n")#registra exito si los totales coinciden
        else:
            f.write("\nESTADO: Operacion Fallida.\n")

# EJECUCIÓN DEL PROCESO
try:#inicia el bloque de control par acapturar posibles errores durante la ejecucion
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")#muestra un mensaje informando  el inicio de conexion con la api externa
    datos = extraer_datos(API_URL)#paso 1: obtencion  de datos

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")#muestra un mensaje informando que se prodcedera a trabajar con la base de datos local SQLite
    guardar_en_db(datos)#paso 2:almacenamiento

    print("Generando archivos de evidencia .CSV y auditoría.txt...")#informa al ususario que ha comenzado la creacion de los archivos fisicos de soportes (csv y txt)
    generar_evidencias(datos)#paso3: verificacion y reportes

    print("¡Proceso completado con éxito!")# informa al usuario que el flujo termino bien
except Exception as e:
    print(f"Ocurrió un error: {e}")# captura y muestra el mensaje de error si algo falla en el "try"

Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



**Referencias**

https://realpython.com/python-requests/#make-a-get-request

https://pandas.pydata.org/

https://pypi.org/project/openpyxl/

https://docs.python.org/3/library/sqlite3.html

https://sqliteviewer.app/

https://sibabalwesinyaniso.medium.com/what-is-a-cursor-understanding-how-sqlite-handles-queries-in-python-8f8c88546820
https://www.pythonlore.com/understanding-sqlite3-cursor-object-for-database-operations/
https://miro.medium.com/v2/resize:fit:720/format:webp/0*rePKZ6jMZbZ_6S_3.gif
